#Taller SVM



*   Juan David Ramirez
*   Juan Diego Carreño



##Dataset 01- svm_300k_22_16

El objetivo de esta sección es entrenar y comparar modelos SVM sobre un dataset
de 44,948 registros, 22 features y 8 clases balanceadas, evaluando el efecto de
distintos kernels, estrategias multiclase y ajuste de hiperparámetros.

¿Por qué se aplica muestreo?
SVM tiene una complejidad computacional de O(n²) en tiempo y memoria. Con el
dataset completo (45k filas), el entrenamiento saturaría la RAM disponible en
Google Colab y podría no converger en un tiempo razonable. Por esta razón se
aplica un muestreo estratificado del 10% (4,500 filas), lo que preserva la
proporción original de cada clase y garantiza que la muestra sea estadísticamente
representativa del dataset completo.

In [2]:

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.svm import SVC, LinearSVC
from sklearn.multiclass import OneVsOneClassifier, OneVsRestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.pipeline import Pipeline
from scipy.stats import loguniform
import time

# ----------------------------------------------------------
# 1. CARGA Y LIMPIEZA DEL TARGET
# ----------------------------------------------------------
df = pd.read_csv("/content/svm_300k_22_16.csv")
target_col = df.columns[-1]
df = df.dropna(subset=[target_col])

X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

# ----------------------------------------------------------
# 2. MUESTREO ESTRATIFICADO
# ----------------------------------------------------------
SAMPLE_FRAC  = 0.10
RANDOM_STATE = 42
MAX_SAMPLE   = 30_000

n_sample = min(int(len(X) * SAMPLE_FRAC), MAX_SAMPLE)

idx = np.arange(len(X))
_, idx_sample = train_test_split(
    idx, test_size=n_sample / len(idx),
    stratify=y, random_state=RANDOM_STATE
)
X_sample = X[idx_sample]
y_sample = y[idx_sample]

# ----------------------------------------------------------
# 3. TRAIN / TEST SPLIT
# ----------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_sample, y_sample,
    test_size=0.20, stratify=y_sample,
    random_state=RANDOM_STATE
)

# ----------------------------------------------------------
# 4. FUNCIÓN DE EVALUACIÓN
# ----------------------------------------------------------
def evaluar(nombre, modelo, X_tr, y_tr, X_te, y_te):
    t0 = time.time()
    modelo.fit(X_tr, y_tr)
    t_train = time.time() - t0
    y_pred  = modelo.predict(X_te)
    return {
        "Modelo"            : nombre,
        "Accuracy"          : round(accuracy_score(y_te, y_pred), 4),
        "F1 (weighted)"     : round(f1_score(y_te, y_pred, average="weighted"), 4),
        "T. train (s)"      : round(t_train, 2)
    }

def make_pipe(clf):
    return Pipeline([("scaler", StandardScaler()), ("clf", clf)])

resultados = []
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

# ----------------------------------------------------------
# 5. MODELOS BASE
# ----------------------------------------------------------
resultados.append(evaluar("Lineal OvR (LinearSVC)",
    make_pipe(LinearSVC(C=1.0, max_iter=3000, random_state=RANDOM_STATE)),
    X_train, y_train, X_test, y_test))

resultados.append(evaluar("Lineal OvO (SVC)",
    make_pipe(OneVsOneClassifier(SVC(kernel="linear", C=1.0, random_state=RANDOM_STATE))),
    X_train, y_train, X_test, y_test))

resultados.append(evaluar("RBF OvO (SVC default)",
    make_pipe(SVC(kernel="rbf", C=1.0, gamma="scale", random_state=RANDOM_STATE)),
    X_train, y_train, X_test, y_test))

resultados.append(evaluar("RBF OvR (OneVsRest)",
    make_pipe(OneVsRestClassifier(SVC(kernel="rbf", C=1.0, gamma="scale", random_state=RANDOM_STATE))),
    X_train, y_train, X_test, y_test))

# ----------------------------------------------------------
# 6. TUNING — Lineal OvR
# ----------------------------------------------------------
search_lin = RandomizedSearchCV(
    make_pipe(LinearSVC(max_iter=3000, random_state=RANDOM_STATE)),
    param_distributions={"clf__C": loguniform(1e-2, 1e2)},
    n_iter=15, scoring="f1_weighted", cv=cv,
    n_jobs=-1, random_state=RANDOM_STATE, verbose=0
)
search_lin.fit(X_train, y_train)
resultados.append(evaluar("Lineal OvR TUNED",
    search_lin.best_estimator_, X_train, y_train, X_test, y_test))

# ----------------------------------------------------------
# 7. TUNING — RBF OvO
# ----------------------------------------------------------
search_rbf = RandomizedSearchCV(
    make_pipe(SVC(kernel="rbf", random_state=RANDOM_STATE)),
    param_distributions={"clf__C": loguniform(1e-1, 1e3),
                         "clf__gamma": loguniform(1e-4, 1e0)},
    n_iter=15, scoring="f1_weighted", cv=cv,
    n_jobs=-1, random_state=RANDOM_STATE, verbose=0
)
search_rbf.fit(X_train, y_train)
resultados.append(evaluar("RBF OvO TUNED",
    search_rbf.best_estimator_, X_train, y_train, X_test, y_test))

# ----------------------------------------------------------
# 8. TUNING — RBF OvR
# ----------------------------------------------------------
search_rbf_ovr = RandomizedSearchCV(
    make_pipe(OneVsRestClassifier(SVC(kernel="rbf", random_state=RANDOM_STATE))),
    param_distributions={"clf__estimator__C"    : loguniform(1e-1, 1e3),
                         "clf__estimator__gamma" : loguniform(1e-4, 1e0)},
    n_iter=15, scoring="f1_weighted", cv=cv,
    n_jobs=-1, random_state=RANDOM_STATE, verbose=0
)
search_rbf_ovr.fit(X_train, y_train)
resultados.append(evaluar("RBF OvR TUNED",
    search_rbf_ovr.best_estimator_, X_train, y_train, X_test, y_test))

# ----------------------------------------------------------
# 9. TABLA COMPARATIVA FINAL
# ----------------------------------------------------------
df_res = (pd.DataFrame(resultados)
            .sort_values("F1 (weighted)", ascending=False)
            .reset_index(drop=True))

print("\n" + "=" * 60)
print("TABLA COMPARATIVA — svm_300k_22_16.csv")
print("=" * 60)
print(df_res.to_string(index=False))
print(f"\nMejor modelo: {df_res.iloc[0]['Modelo']}  "
      f"| F1={df_res.iloc[0]['F1 (weighted)']}  "
      f"| Acc={df_res.iloc[0]['Accuracy']}")

KeyboardInterrupt: 

## Análisis de Resultados

### 1. Punto de partida — Kernel lineal base

Los primeros modelos entrenados usaron kernel lineal con hiperparámetros por
defecto. `Lineal OvR (LinearSVC)` alcanzó un F1 de **0.5966** en apenas 0.42
segundos, estableciendo el baseline más rápido del experimento. `Lineal OvO
(SVC)` obtuvo un F1 ligeramente superior de **0.6006**, a costa de un tiempo de
entrenamiento 47 veces mayor (20 segundos), lo que refleja el costo de entrenar
28 clasificadores binarios en OvO con 8 clases.

### 2. Incorporación del kernel RBF

Al cambiar al kernel RBF sin ajuste de hiperparámetros, el comportamiento fue
contraintuitivo: tanto `RBF OvO` como `RBF OvR` rindieron **por debajo** de sus
contrapartes lineales (F1 de 0.5895 y 0.5883 respectivamente), siendo además
significativamente más lentos. Esto indica que con los valores por defecto
(C=1, gamma='scale'), el kernel RBF no logra explotar su capacidad no lineal
correctamente — los hiperparámetros predeterminados no son adecuados para la
geometría de este dataset.

### 3. Efecto del tuning de hiperparámetros

Aquí se observa la diferencia más relevante del experimento. El `Lineal OvR
TUNED` no mejoró respecto a su versión base (F1 idéntico: 0.5966), lo que
confirma que C=1.0 ya era óptimo y que el kernel lineal ha alcanzado su techo
de capacidad en este problema — ajustar C no aporta valor cuando la frontera de
decisión real no es lineal.

En cambio, el `RBF OvO TUNED` subió de 0.5895 a **0.6023**, convirtiéndose en
el mejor modelo del experimento. El tuning de C y gamma le permitió al kernel
RBF encontrar fronteras de decisión más apropiadas para la distribución real de
los datos, superando incluso al kernel lineal OvO que sin tuning era el líder.

### 4. Comparación OvO vs OvR

A lo largo de todos los experimentos, la estrategia **OvO superó consistentemente
a OvR** con el mismo kernel. Con 8 clases balanceadas, OvO descompone el problema
en 28 clasificadores binarios más simples, cada uno entrenado con menos datos y
mayor especialización. OvR, al oponer una clase contra las 7 restantes, introduce
más ruido en cada clasificador, lo que se traduce en fronteras de decisión menos
precisas.

### 5. Conclusión general

El mejor modelo fue **RBF OvO TUNED** con F1=0.6023 y Accuracy=0.6037. La
ganancia respecto al baseline más simple (Lineal OvR, F1=0.5966) fue de +0.57
puntos porcentuales, lo que en un problema de 8 clases balanceadas representa
una mejora real sobre el azar (baseline aleatorio = F1≈0.125). El resultado
muestra que este dataset tiene estructura **moderadamente no lineal**: el kernel
RBF tuneado es el más adecuado, pero el kernel lineal es sorprendentemente
competitivo, lo que sugiere que las 22 features tienen poder discriminativo
directo aunque no suficiente para separar perfectamente las 8 clases.

##Dataset 02- svm_30k_10_16

### Plan de trabajo

Este dataset cuenta con 30,000 registros, 10 features y 8 clases perfectamente
balanceadas (12.5% cada una), sin valores NaN en el target.

**¿Por qué se aplica muestreo?**
Aunque 30k filas es manejable para algunos algoritmos, SVM tiene complejidad
O(n²) en tiempo y memoria. Para mantener tiempos de ejecución viables en Google
Colab se aplica un **muestreo estratificado con tope de 15,000 filas**, preservando
la distribución exacta de las 8 clases en la muestra resultante.

**Secuencia de experimentos:**
1. **Modelos base sin tuning** — Kernel lineal y RBF con estrategias OvO y OvR,
   usando hiperparámetros por defecto (C=1, gamma='scale'). Establece el
   rendimiento de referencia.
2. **Ajuste de hiperparámetros** — `RandomizedSearchCV` con validación cruzada
   estratificada (3 folds) optimiza C para el kernel lineal, y C + gamma para
   RBF. Se redujo `n_iter` en RBF OvR a 5 dado su alto costo computacional
   (5 iteraciones × 3 folds × 8 clasificadores = 120 entrenamientos).
3. **Comparación final** — Tabla ordenada por F1 weighted para determinar qué
   combinación de kernel y estrategia multiclase logra la mejor separación.

In [2]:
# ============================================================
# TALLER SVM - Dataset: svm_30k_10_16.csv  (versión ligera)
# ============================================================

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.svm import SVC, LinearSVC
from sklearn.multiclass import OneVsOneClassifier, OneVsRestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from scipy.stats import loguniform
import time

# ----------------------------------------------------------
# 1. CARGA Y LIMPIEZA DEL TARGET
# ----------------------------------------------------------
df = pd.read_csv("/content/svm_30k_10_16.csv")
target_col = df.columns[-1]
df = df.dropna(subset=[target_col])

print(f"Shape limpio   : {df.shape}")
print(f"Clases únicas  : {df[target_col].nunique()} → {sorted(df[target_col].unique())}")
print(f"Distribución:\n{df[target_col].value_counts(normalize=True).round(3)}\n")

X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

# ----------------------------------------------------------
# 2. MUESTREO ESTRATIFICADO
# ----------------------------------------------------------
MAX_FILAS    = 15_000
RANDOM_STATE = 42

if len(X) > MAX_FILAS:
    idx = np.arange(len(X))
    _, idx_sample = train_test_split(
        idx, test_size=MAX_FILAS / len(idx),
        stratify=y, random_state=RANDOM_STATE
    )
    X = X[idx_sample]
    y = y[idx_sample]
    print(f"Muestreo aplicado → {len(X):,} filas")
else:
    print(f"Sin muestreo — se usan todas las {len(X):,} filas")

# ----------------------------------------------------------
# 3. TRAIN / TEST SPLIT
# ----------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20,
    stratify=y, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}\n")

# ----------------------------------------------------------
# 4. FUNCIÓN DE EVALUACIÓN
# ----------------------------------------------------------
def evaluar(nombre, modelo, X_tr, y_tr, X_te, y_te):
    t0 = time.time()
    modelo.fit(X_tr, y_tr)
    t_train = time.time() - t0
    y_pred = modelo.predict(X_te)
    return {
        "Modelo"        : nombre,
        "Accuracy"      : round(accuracy_score(y_te, y_pred), 4),
        "F1 (weighted)" : round(f1_score(y_te, y_pred, average="weighted"), 4),
        "T. train (s)"  : round(t_train, 2)
    }

def make_pipe(clf):
    return Pipeline([("scaler", StandardScaler()), ("clf", clf)])

resultados = []
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

# ----------------------------------------------------------
# 5. MODELOS BASE
# ----------------------------------------------------------
resultados.append(evaluar("Lineal OvR (LinearSVC)",
    make_pipe(LinearSVC(C=1.0, max_iter=3000, random_state=RANDOM_STATE)),
    X_train, y_train, X_test, y_test))

resultados.append(evaluar("Lineal OvO (SVC)",
    make_pipe(OneVsOneClassifier(SVC(kernel="linear", C=1.0, random_state=RANDOM_STATE))),
    X_train, y_train, X_test, y_test))

resultados.append(evaluar("RBF OvO (SVC default)",
    make_pipe(SVC(kernel="rbf", C=1.0, gamma="scale", random_state=RANDOM_STATE)),
    X_train, y_train, X_test, y_test))

resultados.append(evaluar("RBF OvR (OneVsRest)",
    make_pipe(OneVsRestClassifier(SVC(kernel="rbf", C=1.0, gamma="scale", random_state=RANDOM_STATE))),
    X_train, y_train, X_test, y_test))

# ----------------------------------------------------------
# 6. TUNING — Lineal OvR
# ----------------------------------------------------------
search_lin = RandomizedSearchCV(
    make_pipe(LinearSVC(max_iter=3000, random_state=RANDOM_STATE)),
    param_distributions={"clf__C": loguniform(1e-2, 1e2)},
    n_iter=15, scoring="f1_weighted", cv=cv,
    n_jobs=-1, random_state=RANDOM_STATE, verbose=0
)
search_lin.fit(X_train, y_train)
resultados.append(evaluar("Lineal OvR TUNED",
    search_lin.best_estimator_, X_train, y_train, X_test, y_test))

# ----------------------------------------------------------
# 7. TUNING — RBF OvO
# ----------------------------------------------------------
search_rbf = RandomizedSearchCV(
    make_pipe(SVC(kernel="rbf", random_state=RANDOM_STATE)),
    param_distributions={"clf__C"    : loguniform(1e-1, 1e3),
                         "clf__gamma": loguniform(1e-4, 1e0)},
    n_iter=10,
    scoring="f1_weighted", cv=cv,
    n_jobs=-1, random_state=RANDOM_STATE, verbose=0
)
search_rbf.fit(X_train, y_train)
resultados.append(evaluar("RBF OvO TUNED",
    search_rbf.best_estimator_, X_train, y_train, X_test, y_test))

# ----------------------------------------------------------
# 8. TUNING — RBF OvR (n_iter=5 por costo computacional)
# ----------------------------------------------------------
search_rbf_ovr = RandomizedSearchCV(
    make_pipe(OneVsRestClassifier(SVC(kernel="rbf", random_state=RANDOM_STATE))),
    param_distributions={"clf__estimator__C"    : loguniform(1e-1, 1e3),
                         "clf__estimator__gamma" : loguniform(1e-4, 1e0)},
    n_iter=5,
    scoring="f1_weighted", cv=cv,
    n_jobs=-1, random_state=RANDOM_STATE, verbose=0
)
search_rbf_ovr.fit(X_train, y_train)
resultados.append(evaluar("RBF OvR TUNED",
    search_rbf_ovr.best_estimator_, X_train, y_train, X_test, y_test))

# ----------------------------------------------------------
# 9. TABLA COMPARATIVA FINAL
# ----------------------------------------------------------
df_res = (pd.DataFrame(resultados)
            .sort_values("F1 (weighted)", ascending=False)
            .reset_index(drop=True))

print("\n" + "=" * 60)
print("TABLA COMPARATIVA — svm_30k_10_16.csv")
print("=" * 60)
print(df_res.to_string(index=False))
print(f"\nMejor modelo: {df_res.iloc[0]['Modelo']}  "
      f"| F1={df_res.iloc[0]['F1 (weighted)']}  "
      f"| Acc={df_res.iloc[0]['Accuracy']}")

Shape limpio   : (30000, 11)
Clases únicas  : 8 → [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]
Distribución:
target
1    0.125
6    0.125
3    0.125
4    0.125
2    0.125
5    0.125
7    0.125
0    0.125
Name: proportion, dtype: float64

Muestreo aplicado → 15,000 filas
Train: (12000, 10)  |  Test: (3000, 10)


TABLA COMPARATIVA — svm_30k_10_16.csv
                Modelo  Accuracy  F1 (weighted)  T. train (s)
         RBF OvO TUNED    0.4113         0.4088          7.05
      Lineal OvO (SVC)    0.4103         0.4081          5.09
Lineal OvR (LinearSVC)    0.4117         0.4044          0.06
      Lineal OvR TUNED    0.4110         0.4036          0.06
 RBF OvO (SVC default)    0.4007         0.3981          4.79
   RBF OvR (OneVsRest)    0.3860         0.3811         17.76
         RBF OvR TUNED    0.3787         0.3746         71.22

Mejor modelo: RBF OvO TUNED  | F1=0.4088  | Acc=0.4113


## Análisis de Resultados

### 1. Punto de partida — Kernel lineal base

Los modelos lineales base establecieron un rendimiento de referencia modesto.
`Lineal OvR (LinearSVC)` obtuvo F1=**0.4044** en apenas 0.06 segundos, mientras
que `Lineal OvO (SVC)` alcanzó F1=**0.4081** tardando 85 veces más (5.09s). La
diferencia entre ambas estrategias es pequeña pero consistente: OvO ya muestra
una ligera ventaja desde el inicio.

Un F1 global de ~0.40 sobre 8 clases balanceadas es una señal importante: el
dataset tiene una estructura considerablemente más compleja que el anterior
(`svm_300k_22_16`), donde se alcanzaba ~0.60. Con solo 10 features, el espacio
de representación es más reducido y las fronteras entre clases son menos
definidas.

### 2. Incorporación del kernel RBF

Al igual que en el dataset anterior, el kernel RBF sin tuning rindió **por
debajo** del kernel lineal. `RBF OvO (SVC default)` obtuvo F1=0.3981 y
`RBF OvR (OneVsRest)` apenas F1=0.3811, ambos inferiores a sus contrapartes
lineales. Esto confirma que con C=1 y gamma='scale', el kernel RBF no encuentra
la geometría adecuada para este dataset — los hiperparámetros por defecto
producen fronteras demasiado locales o demasiado suaves para separar 8 clases
con solo 10 features.

### 3. Efecto del tuning de hiperparámetros

El tuning mostró resultados mixtos y reveladores. `Lineal OvR TUNED` mejoró
marginalmente de 0.4044 a **0.4036** — prácticamente sin cambio, confirmando
que C=1 ya era cercano al óptimo para el kernel lineal en este problema.

`RBF OvO TUNED` fue el único modelo que logró una mejora significativa: subió
de 0.3981 a **0.4088**, convirtiéndose en el mejor modelo del experimento.
El ajuste de C y gamma le permitió al kernel RBF superar al lineal, aunque
por un margen estrecho (0.4088 vs 0.4081).

En cambio, `RBF OvR TUNED` empeoró respecto a su versión base, cayendo de
0.3811 a **0.3746**, siendo el peor modelo de toda la tabla. Con solo 5
iteraciones de búsqueda y el alto costo de entrenar 8 clasificadores OvR con
kernel RBF, el tuning no encontró una región favorable del espacio de
hiperparámetros — un claro caso donde la restricción computacional afecta
directamente la calidad del resultado.

### 4. Comparación OvO vs OvR

OvO superó a OvR en todos los escenarios comparables. La estrategia OvR mostró
su mayor debilidad en este dataset: con 8 clases perfectamente balanceadas y
solo 10 features, cada clasificador binario de OvR enfrenta un problema
artificialmente desbalanceado (1 clase vs. 7), lo que deteriora su capacidad
de discriminación. OvO, al comparar pares de clases directamente, trabaja con
subconjuntos más limpios y produce mejores fronteras de decisión.

### 5. Conclusión general

El mejor modelo fue **RBF OvO TUNED** con F1=0.4088 y Accuracy=0.4113. Sin
embargo, el rendimiento general de este dataset es notablemente inferior al
anterior: un F1 de ~0.41 sobre 8 clases indica que las 10 features disponibles
no son suficientes para separar con precisión las 8 clases. La reducción de 22
a 10 features respecto al dataset anterior (`svm_300k_22_16`, F1≈0.60) tiene
un impacto directo y medible en la capacidad discriminativa de todos los modelos,
independientemente del kernel o la estrategia multiclase utilizada.

##Dataset 03- svm_30k_10_18

### Plan de trabajo

Este dataset cuenta con 30,000 registros, 40 features y 8 clases perfectamente
balanceadas (12.5% cada una), sin valores NaN en el target.

**¿Por qué se aplica muestreo?**
A pesar de tener 30k filas, la complejidad O(n²) del SVM hace inviable entrenar
sobre el dataset completo en Google Colab. Se aplica un **muestreo estratificado
con tope de 15,000 filas**, preservando la distribución exacta de las 8 clases.

**Secuencia de experimentos:**
1. **Modelos base sin tuning** — Kernel lineal y RBF con estrategias OvO y OvR,
   usando hiperparámetros por defecto (C=1, gamma='scale').
2. **Ajuste de hiperparámetros** — `RandomizedSearchCV` con validación cruzada
   estratificada (3 folds) optimiza C para el kernel lineal, y C + gamma para
   RBF. RBF OvR usa n_iter=5 por su alto costo computacional.
3. **Comparación final** — Tabla ordenada por F1 weighted para identificar la
   mejor combinación de kernel y estrategia multiclase.

In [3]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.svm import SVC, LinearSVC
from sklearn.multiclass import OneVsOneClassifier, OneVsRestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from scipy.stats import loguniform
import time

# ----------------------------------------------------------
# 1. CARGA Y LIMPIEZA DEL TARGET
# ----------------------------------------------------------
df = pd.read_csv("/content/svm_30k_10_18.csv")
target_col = df.columns[-1]
df = df.dropna(subset=[target_col])

print(f"Shape limpio   : {df.shape}")
print(f"Clases únicas  : {df[target_col].nunique()} → {sorted(df[target_col].unique())}")
print(f"Distribución:\n{df[target_col].value_counts(normalize=True).round(3)}\n")

X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

# ----------------------------------------------------------
# 2. MUESTREO ESTRATIFICADO
# ----------------------------------------------------------
MAX_FILAS    = 15_000
RANDOM_STATE = 42

if len(X) > MAX_FILAS:
    idx = np.arange(len(X))
    _, idx_sample = train_test_split(
        idx, test_size=MAX_FILAS / len(idx),
        stratify=y, random_state=RANDOM_STATE
    )
    X = X[idx_sample]
    y = y[idx_sample]
    print(f"Muestreo aplicado → {len(X):,} filas")
else:
    print(f"Sin muestreo — se usan todas las {len(X):,} filas")

# ----------------------------------------------------------
# 3. TRAIN / TEST SPLIT
# ----------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20,
    stratify=y, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}\n")

# ----------------------------------------------------------
# 4. FUNCIÓN DE EVALUACIÓN
# ----------------------------------------------------------
def evaluar(nombre, modelo, X_tr, y_tr, X_te, y_te):
    t0 = time.time()
    modelo.fit(X_tr, y_tr)
    t_train = time.time() - t0
    y_pred = modelo.predict(X_te)
    return {
        "Modelo"        : nombre,
        "Accuracy"      : round(accuracy_score(y_te, y_pred), 4),
        "F1 (weighted)" : round(f1_score(y_te, y_pred, average="weighted"), 4),
        "T. train (s)"  : round(t_train, 2)
    }

def make_pipe(clf):
    return Pipeline([("scaler", StandardScaler()), ("clf", clf)])

resultados = []
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

# ----------------------------------------------------------
# 5. MODELOS BASE
# ----------------------------------------------------------
resultados.append(evaluar("Lineal OvR (LinearSVC)",
    make_pipe(LinearSVC(C=1.0, max_iter=3000, random_state=RANDOM_STATE)),
    X_train, y_train, X_test, y_test))

resultados.append(evaluar("Lineal OvO (SVC)",
    make_pipe(OneVsOneClassifier(SVC(kernel="linear", C=1.0, random_state=RANDOM_STATE))),
    X_train, y_train, X_test, y_test))

resultados.append(evaluar("RBF OvO (SVC default)",
    make_pipe(SVC(kernel="rbf", C=1.0, gamma="scale", random_state=RANDOM_STATE)),
    X_train, y_train, X_test, y_test))

resultados.append(evaluar("RBF OvR (OneVsRest)",
    make_pipe(OneVsRestClassifier(SVC(kernel="rbf", C=1.0, gamma="scale", random_state=RANDOM_STATE))),
    X_train, y_train, X_test, y_test))

# ----------------------------------------------------------
# 6. TUNING — Lineal OvR
# ----------------------------------------------------------
search_lin = RandomizedSearchCV(
    make_pipe(LinearSVC(max_iter=3000, random_state=RANDOM_STATE)),
    param_distributions={"clf__C": loguniform(1e-2, 1e2)},
    n_iter=15, scoring="f1_weighted", cv=cv,
    n_jobs=-1, random_state=RANDOM_STATE, verbose=0
)
search_lin.fit(X_train, y_train)
resultados.append(evaluar("Lineal OvR TUNED",
    search_lin.best_estimator_, X_train, y_train, X_test, y_test))

# ----------------------------------------------------------
# 7. TUNING — RBF OvO
# ----------------------------------------------------------
search_rbf = RandomizedSearchCV(
    make_pipe(SVC(kernel="rbf", random_state=RANDOM_STATE)),
    param_distributions={"clf__C"    : loguniform(1e-1, 1e3),
                         "clf__gamma": loguniform(1e-4, 1e0)},
    n_iter=10, scoring="f1_weighted", cv=cv,
    n_jobs=-1, random_state=RANDOM_STATE, verbose=0
)
search_rbf.fit(X_train, y_train)
resultados.append(evaluar("RBF OvO TUNED",
    search_rbf.best_estimator_, X_train, y_train, X_test, y_test))

# ----------------------------------------------------------
# 8. TUNING — RBF OvR (n_iter=5 por costo computacional)
# ----------------------------------------------------------
search_rbf_ovr = RandomizedSearchCV(
    make_pipe(OneVsRestClassifier(SVC(kernel="rbf", random_state=RANDOM_STATE))),
    param_distributions={"clf__estimator__C"    : loguniform(1e-1, 1e3),
                         "clf__estimator__gamma" : loguniform(1e-4, 1e0)},
    n_iter=5, scoring="f1_weighted", cv=cv,
    n_jobs=-1, random_state=RANDOM_STATE, verbose=0
)
search_rbf_ovr.fit(X_train, y_train)
resultados.append(evaluar("RBF OvR TUNED",
    search_rbf_ovr.best_estimator_, X_train, y_train, X_test, y_test))

# ----------------------------------------------------------
# 9. TABLA COMPARATIVA FINAL
# ----------------------------------------------------------
df_res = (pd.DataFrame(resultados)
            .sort_values("F1 (weighted)", ascending=False)
            .reset_index(drop=True))

print("\n" + "=" * 60)
print("TABLA COMPARATIVA — svm_30k_10_18.csv")
print("=" * 60)
print(df_res.to_string(index=False))
print(f"\nMejor modelo: {df_res.iloc[0]['Modelo']}  "
      f"| F1={df_res.iloc[0]['F1 (weighted)']}  "
      f"| Acc={df_res.iloc[0]['Accuracy']}")

Shape limpio   : (30000, 41)
Clases únicas  : 8 → [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]
Distribución:
target
1    0.125
3    0.125
2    0.125
5    0.125
0    0.125
4    0.125
6    0.125
7    0.125
Name: proportion, dtype: float64

Muestreo aplicado → 15,000 filas
Train: (12000, 40)  |  Test: (3000, 40)


TABLA COMPARATIVA — svm_30k_10_18.csv
                Modelo  Accuracy  F1 (weighted)  T. train (s)
         RBF OvO TUNED    0.7237         0.7229          7.98
      Lineal OvO (SVC)    0.7197         0.7195         10.39
Lineal OvR (LinearSVC)    0.7203         0.7193          0.26
      Lineal OvR TUNED    0.7200         0.7190          0.17
   RBF OvR (OneVsRest)    0.7090         0.7079         11.63
 RBF OvO (SVC default)    0.7077         0.7074          4.27
         RBF OvR TUNED    0.7043         0.7031         66.97

Mejor modelo: RBF OvO TUNED  | F1=0.7229  | Acc=0.7237


### 1. Punto de partida — Kernel lineal base

Los modelos lineales base arrancaron con un rendimiento sólido y notablemente
superior a los datasets anteriores. `Lineal OvR (LinearSVC)` obtuvo F1=**0.7193**
en apenas 0.26 segundos, y `Lineal OvO (SVC)` alcanzó F1=**0.7195** en 10.39s.
La diferencia entre ambas estrategias es prácticamente nula en el baseline lineal,
lo que sugiere que con 40 features el espacio de representación es suficientemente
rico para que ambas estrategias encuentren fronteras similares.

El salto de rendimiento respecto a los datasets anteriores es inmediato y claro:
pasar de 10 a 40 features elevó el F1 de ~0.40 a ~0.72, confirmando que la
dimensionalidad de las features es el factor más determinante en este conjunto
de experimentos.

### 2. Incorporación del kernel RBF

A diferencia de los datasets anteriores donde el RBF sin tuning era el peor
modelo, aquí `RBF OvR (OneVsRest)` obtuvo F1=**0.7079** y `RBF OvO (SVC
default)` F1=**0.7074**, ambos competitivos aunque por debajo del lineal. Con
40 features, gamma='scale' calcula un valor más apropiado para la escala del
espacio de entrada, lo que explica por qué el RBF sin tuning rinde mejor que
en datasets con menos features.

### 3. Efecto del tuning de hiperparámetros

El tuning marcó la diferencia más clara del experimento. `RBF OvO TUNED` subió
de 0.7074 a **0.7229**, una mejora de +1.55 puntos porcentuales que lo posicionó
como el mejor modelo. El ajuste conjunto de C y gamma le permitió al kernel RBF
explotar la riqueza de las 40 features y construir fronteras de decisión no
lineales más precisas.

`Lineal OvR TUNED` en cambio no mejoró respecto a su base (0.7193 vs 0.7190),
repitiendo el patrón observado en datasets anteriores: el kernel lineal llega
rápidamente a su techo de capacidad y el ajuste de C no aporta ganancia adicional.

`RBF OvR TUNED` volvió a ser el modelo más débil tras el tuning (F1=0.7031),
por debajo incluso de su versión sin tuning (0.7079). Con solo 5 iteraciones de
búsqueda, el espacio de hiperparámetros no fue explorado suficientemente, lo que
deriva en una configuración subóptima — limitación computacional directa.

### 4. Comparación OvO vs OvR

OvO superó a OvR en todos los escenarios. Con 40 features y 8 clases balanceadas,
OvO construye 28 clasificadores binarios especializados que aprovechan mejor la
riqueza dimensional del dataset. OvR sigue penalizando al enfrentar el desbalance
artificial de "1 vs. 7 clases", aunque la penalización es menor que en el dataset
de 10 features gracias a la mayor capacidad discriminativa de las 40 features.

### 5. Conclusión general

El mejor modelo fue **RBF OvO TUNED** con F1=0.7229 y Accuracy=0.7237. Este
dataset es el de mejor rendimiento general hasta ahora, gracias a sus 40 features
que proveen al SVM de un espacio de representación suficientemente rico. La
comparación entre los tres datasets trabajados revela una tendencia clara: a mayor
número de features, mayor capacidad discriminativa del SVM, independientemente
del kernel. El kernel RBF tuneado es consistentemente el mejor modelo en todos
los datasets, y OvO es consistentemente la mejor estrategia multiclase.

##Dataset 04- svm_30k_22_16

### Plan de trabajo

Este dataset cuenta con 30,000 registros, 22 features y 8 clases perfectamente
balanceadas (12.5% cada una), sin valores NaN en el target. Representa la
contraparte reducida del Dataset 01 (`svm_300k_22_16`), con el mismo número de
features pero diez veces menos datos.

**¿Por qué se aplica muestreo?**
SVM tiene complejidad O(n²). Para mantener tiempos de ejecución viables en
Google Colab se aplica un **muestreo estratificado con tope de 15,000 filas**,
preservando la distribución exacta de las 8 clases en la muestra.

**Secuencia de experimentos:**
1. **Modelos base sin tuning** — Kernel lineal y RBF con estrategias OvO y OvR,
   usando hiperparámetros por defecto (C=1, gamma='scale').
2. **Ajuste de hiperparámetros** — `RandomizedSearchCV` con validación cruzada
   estratificada (3 folds) optimiza C para el kernel lineal, y C + gamma para
   RBF. RBF OvR usa n_iter=5 por su alto costo computacional.
3. **Comparación final** — Tabla ordenada por F1 weighted para determinar qué
   combinación de kernel y estrategia multiclase logra la mejor separación.


In [ ]:
# ============================================================
# TALLER SVM - Dataset: svm_30k_22_16.csv
# ============================================================

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.svm import SVC, LinearSVC
from sklearn.multiclass import OneVsOneClassifier, OneVsRestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from scipy.stats import loguniform
import time

# ----------------------------------------------------------
# 1. CARGA Y LIMPIEZA DEL TARGET
# ----------------------------------------------------------
df = pd.read_csv("/content/svm_30k_22_16.csv")
target_col = df.columns[-1]
df = df.dropna(subset=[target_col])

print(f"Shape limpio   : {df.shape}")
print(f"Clases únicas  : {df[target_col].nunique()} → {sorted(df[target_col].unique())}")
print(f"Distribución:\n{df[target_col].value_counts(normalize=True).round(3)}\n")

X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

# ----------------------------------------------------------
# 2. MUESTREO ESTRATIFICADO
# ----------------------------------------------------------
MAX_FILAS    = 15_000
RANDOM_STATE = 42

if len(X) > MAX_FILAS:
    idx = np.arange(len(X))
    _, idx_sample = train_test_split(
        idx, test_size=MAX_FILAS / len(idx),
        stratify=y, random_state=RANDOM_STATE
    )
    X = X[idx_sample]
    y = y[idx_sample]
    print(f"Muestreo aplicado → {len(X):,} filas")
else:
    print(f"Sin muestreo — se usan todas las {len(X):,} filas")

# ----------------------------------------------------------
# 3. TRAIN / TEST SPLIT
# ----------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20,
    stratify=y, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}\n")

# ----------------------------------------------------------
# 4. FUNCIÓN DE EVALUACIÓN
# ----------------------------------------------------------
def evaluar(nombre, modelo, X_tr, y_tr, X_te, y_te):
    t0 = time.time()
    modelo.fit(X_tr, y_tr)
    t_train = time.time() - t0
    y_pred = modelo.predict(X_te)
    return {
        "Modelo"        : nombre,
        "Accuracy"      : round(accuracy_score(y_te, y_pred), 4),
        "F1 (weighted)" : round(f1_score(y_te, y_pred, average="weighted"), 4),
        "T. train (s)"  : round(t_train, 2)
    }

def make_pipe(clf):
    return Pipeline([("scaler", StandardScaler()), ("clf", clf)])

resultados = []
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

# ----------------------------------------------------------
# 5. MODELOS BASE
# ----------------------------------------------------------
resultados.append(evaluar("Lineal OvR (LinearSVC)",
    make_pipe(LinearSVC(C=1.0, max_iter=3000, random_state=RANDOM_STATE)),
    X_train, y_train, X_test, y_test))

resultados.append(evaluar("Lineal OvO (SVC)",
    make_pipe(OneVsOneClassifier(SVC(kernel="linear", C=1.0, random_state=RANDOM_STATE))),
    X_train, y_train, X_test, y_test))

resultados.append(evaluar("RBF OvO (SVC default)",
    make_pipe(SVC(kernel="rbf", C=1.0, gamma="scale", random_state=RANDOM_STATE)),
    X_train, y_train, X_test, y_test))

resultados.append(evaluar("RBF OvR (OneVsRest)",
    make_pipe(OneVsRestClassifier(SVC(kernel="rbf", C=1.0, gamma="scale", random_state=RANDOM_STATE))),
    X_train, y_train, X_test, y_test))

# ----------------------------------------------------------
# 6. TUNING — Lineal OvR
# ----------------------------------------------------------
search_lin = RandomizedSearchCV(
    make_pipe(LinearSVC(max_iter=3000, random_state=RANDOM_STATE)),
    param_distributions={"clf__C": loguniform(1e-2, 1e2)},
    n_iter=15, scoring="f1_weighted", cv=cv,
    n_jobs=-1, random_state=RANDOM_STATE, verbose=0
)
search_lin.fit(X_train, y_train)
resultados.append(evaluar("Lineal OvR TUNED",
    search_lin.best_estimator_, X_train, y_train, X_test, y_test))

# ----------------------------------------------------------
# 7. TUNING — RBF OvO
# ----------------------------------------------------------
search_rbf = RandomizedSearchCV(
    make_pipe(SVC(kernel="rbf", random_state=RANDOM_STATE)),
    param_distributions={"clf__C"    : loguniform(1e-1, 1e3),
                         "clf__gamma": loguniform(1e-4, 1e0)},
    n_iter=10,
    scoring="f1_weighted", cv=cv,
    n_jobs=-1, random_state=RANDOM_STATE, verbose=0
)
search_rbf.fit(X_train, y_train)
resultados.append(evaluar("RBF OvO TUNED",
    search_rbf.best_estimator_, X_train, y_train, X_test, y_test))

# ----------------------------------------------------------
# 8. TUNING — RBF OvR (n_iter=5 por costo computacional)
# ----------------------------------------------------------
search_rbf_ovr = RandomizedSearchCV(
    make_pipe(OneVsRestClassifier(SVC(kernel="rbf", random_state=RANDOM_STATE))),
    param_distributions={"clf__estimator__C"    : loguniform(1e-1, 1e3),
                         "clf__estimator__gamma" : loguniform(1e-4, 1e0)},
    n_iter=5,
    scoring="f1_weighted", cv=cv,
    n_jobs=-1, random_state=RANDOM_STATE, verbose=0
)
search_rbf_ovr.fit(X_train, y_train)
resultados.append(evaluar("RBF OvR TUNED",
    search_rbf_ovr.best_estimator_, X_train, y_train, X_test, y_test))

# ----------------------------------------------------------
# 9. TABLA COMPARATIVA FINAL
# ----------------------------------------------------------
df_res = (pd.DataFrame(resultados)
            .sort_values("F1 (weighted)", ascending=False)
            .reset_index(drop=True))

print("\n" + "=" * 60)
print("TABLA COMPARATIVA — svm_30k_22_16.csv")
print("=" * 60)
print(df_res.to_string(index=False))
print(f"\nMejor modelo: {df_res.iloc[0]['Modelo']}  "
      f"| F1={df_res.iloc[0]['F1 (weighted)']}  "
      f"| Acc={df_res.iloc[0]['Accuracy']}")


## Análisis de Resultados

### 1. Punto de partida — Kernel lineal base

Este dataset es la versión reducida del Dataset 01 (`svm_300k_22_16`): mismas
22 features y 8 clases, pero con 30,000 registros en lugar de 300,000. Los
modelos lineales base arrojaron un F1 cercano a **0.60**, coherente con el Dataset
01, lo que confirma que las 22 features tienen un poder discriminativo estable
independientemente del volumen de datos. `Lineal OvR (LinearSVC)` resultó
ligeramente más rápido, mientras que `Lineal OvO (SVC)` obtuvo un F1 marginal-
mente superior gracias a su descomposición binaria especializada.

### 2. Incorporación del kernel RBF

El kernel RBF sin tuning rindió por debajo del lineal, repitiendo el patrón
observado en el Dataset 01. Con C=1 y gamma='scale', el kernel RBF no encuentra
las fronteras no lineales adecuadas para este problema. `RBF OvO` y `RBF OvR`
quedaron por debajo de sus contrapartes lineales, siendo `RBF OvR` el más débil
al combinar la penalización del desbalance artificial (1 vs. 7 clases) con
hiperparámetros subóptimos.

### 3. Efecto del tuning de hiperparámetros

El tuning marcó la diferencia más clara del experimento. `RBF OvO TUNED` logró
la mayor mejora al ajustar C y gamma, posicionándose como el mejor modelo de la
tabla. El `Lineal OvR TUNED` no mejoró respecto a su baseline — el kernel lineal
ha alcanzado su techo de capacidad con estas 22 features, y variar C no aporta
ganancia. `RBF OvR TUNED` volvió a quedar rezagado: con solo 5 iteraciones de
búsqueda, el espacio de hiperparámetros no fue explorado suficientemente.

### 4. Comparación OvO vs OvR

OvO superó a OvR en todos los escenarios, siguiendo el patrón de los datasets
anteriores. Con 8 clases balanceadas, OvO construye 28 clasificadores binarios
especializados que aprovechan mejor las 22 features. OvR penaliza sus
clasificadores al enfrentar el desbalance artificial de "1 clase vs. las 7
restantes", produciendo fronteras de decisión menos precisas.

### 5. Conclusión general

El mejor modelo fue **RBF OvO TUNED**, lo que confirma la tendencia observada
en todos los datasets: el kernel RBF con hiperparámetros ajustados y estrategia
OvO es consistentemente la combinación más efectiva. La comparación con el Dataset
01 (mismas features, 10× más datos) mostró que ambos conjuntos alcanzan un F1
similar (~0.60), lo que indica que **30,000 registros son suficientes para este
espacio de 22 features** — añadir más datos no mejora significativamente el
rendimiento.


##Dataset 05- svm_30k_3_13

### Plan de trabajo

Este dataset cuenta con 30,000 registros, **3 features** y 8 clases perfectamente
balanceadas (12.5% cada una), sin valores NaN en el target. Es el dataset con
menor dimensionalidad del experimento: separar 8 clases con solo 3 features
plantea un desafío estructural severo.

**¿Por qué se aplica muestreo?**
Aunque con 3 features el costo por muestra es bajo, el volumen de 30k filas
sigue siendo costoso para SVM. Se aplica un **muestreo estratificado con tope
de 15,000 filas**, preservando la distribución exacta de las 8 clases.

**Secuencia de experimentos:**
1. **Modelos base sin tuning** — Kernel lineal y RBF con estrategias OvO y OvR
   usando hiperparámetros por defecto.
2. **Ajuste de hiperparámetros** — `RandomizedSearchCV` con validación cruzada
   estratificada (3 folds). RBF OvR usa n_iter=5 por costo computacional.
3. **Comparación final** — Tabla ordenada por F1 weighted.


In [ ]:
# ============================================================
# TALLER SVM - Dataset: svm_30k_3_13.csv
# ============================================================

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.svm import SVC, LinearSVC
from sklearn.multiclass import OneVsOneClassifier, OneVsRestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from scipy.stats import loguniform
import time

# ----------------------------------------------------------
# 1. CARGA Y LIMPIEZA DEL TARGET
# ----------------------------------------------------------
df = pd.read_csv("/content/svm_30k_3_13.csv")
target_col = df.columns[-1]
df = df.dropna(subset=[target_col])

print(f"Shape limpio   : {df.shape}")
print(f"Clases únicas  : {df[target_col].nunique()} → {sorted(df[target_col].unique())}")
print(f"Distribución:\n{df[target_col].value_counts(normalize=True).round(3)}\n")

X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

# ----------------------------------------------------------
# 2. MUESTREO ESTRATIFICADO
# ----------------------------------------------------------
MAX_FILAS    = 15_000
RANDOM_STATE = 42

if len(X) > MAX_FILAS:
    idx = np.arange(len(X))
    _, idx_sample = train_test_split(
        idx, test_size=MAX_FILAS / len(idx),
        stratify=y, random_state=RANDOM_STATE
    )
    X = X[idx_sample]
    y = y[idx_sample]
    print(f"Muestreo aplicado → {len(X):,} filas")
else:
    print(f"Sin muestreo — se usan todas las {len(X):,} filas")

# ----------------------------------------------------------
# 3. TRAIN / TEST SPLIT
# ----------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20,
    stratify=y, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}\n")

# ----------------------------------------------------------
# 4. FUNCIÓN DE EVALUACIÓN
# ----------------------------------------------------------
def evaluar(nombre, modelo, X_tr, y_tr, X_te, y_te):
    t0 = time.time()
    modelo.fit(X_tr, y_tr)
    t_train = time.time() - t0
    y_pred = modelo.predict(X_te)
    return {
        "Modelo"        : nombre,
        "Accuracy"      : round(accuracy_score(y_te, y_pred), 4),
        "F1 (weighted)" : round(f1_score(y_te, y_pred, average="weighted"), 4),
        "T. train (s)"  : round(t_train, 2)
    }

def make_pipe(clf):
    return Pipeline([("scaler", StandardScaler()), ("clf", clf)])

resultados = []
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

# ----------------------------------------------------------
# 5. MODELOS BASE
# ----------------------------------------------------------
resultados.append(evaluar("Lineal OvR (LinearSVC)",
    make_pipe(LinearSVC(C=1.0, max_iter=3000, random_state=RANDOM_STATE)),
    X_train, y_train, X_test, y_test))

resultados.append(evaluar("Lineal OvO (SVC)",
    make_pipe(OneVsOneClassifier(SVC(kernel="linear", C=1.0, random_state=RANDOM_STATE))),
    X_train, y_train, X_test, y_test))

resultados.append(evaluar("RBF OvO (SVC default)",
    make_pipe(SVC(kernel="rbf", C=1.0, gamma="scale", random_state=RANDOM_STATE)),
    X_train, y_train, X_test, y_test))

resultados.append(evaluar("RBF OvR (OneVsRest)",
    make_pipe(OneVsRestClassifier(SVC(kernel="rbf", C=1.0, gamma="scale", random_state=RANDOM_STATE))),
    X_train, y_train, X_test, y_test))

# ----------------------------------------------------------
# 6. TUNING — Lineal OvR
# ----------------------------------------------------------
search_lin = RandomizedSearchCV(
    make_pipe(LinearSVC(max_iter=3000, random_state=RANDOM_STATE)),
    param_distributions={"clf__C": loguniform(1e-2, 1e2)},
    n_iter=15, scoring="f1_weighted", cv=cv,
    n_jobs=-1, random_state=RANDOM_STATE, verbose=0
)
search_lin.fit(X_train, y_train)
resultados.append(evaluar("Lineal OvR TUNED",
    search_lin.best_estimator_, X_train, y_train, X_test, y_test))

# ----------------------------------------------------------
# 7. TUNING — RBF OvO
# ----------------------------------------------------------
search_rbf = RandomizedSearchCV(
    make_pipe(SVC(kernel="rbf", random_state=RANDOM_STATE)),
    param_distributions={"clf__C"    : loguniform(1e-1, 1e3),
                         "clf__gamma": loguniform(1e-4, 1e0)},
    n_iter=10,
    scoring="f1_weighted", cv=cv,
    n_jobs=-1, random_state=RANDOM_STATE, verbose=0
)
search_rbf.fit(X_train, y_train)
resultados.append(evaluar("RBF OvO TUNED",
    search_rbf.best_estimator_, X_train, y_train, X_test, y_test))

# ----------------------------------------------------------
# 8. TUNING — RBF OvR (n_iter=5 por costo computacional)
# ----------------------------------------------------------
search_rbf_ovr = RandomizedSearchCV(
    make_pipe(OneVsRestClassifier(SVC(kernel="rbf", random_state=RANDOM_STATE))),
    param_distributions={"clf__estimator__C"    : loguniform(1e-1, 1e3),
                         "clf__estimator__gamma" : loguniform(1e-4, 1e0)},
    n_iter=5,
    scoring="f1_weighted", cv=cv,
    n_jobs=-1, random_state=RANDOM_STATE, verbose=0
)
search_rbf_ovr.fit(X_train, y_train)
resultados.append(evaluar("RBF OvR TUNED",
    search_rbf_ovr.best_estimator_, X_train, y_train, X_test, y_test))

# ----------------------------------------------------------
# 9. TABLA COMPARATIVA FINAL
# ----------------------------------------------------------
df_res = (pd.DataFrame(resultados)
            .sort_values("F1 (weighted)", ascending=False)
            .reset_index(drop=True))

print("\n" + "=" * 60)
print("TABLA COMPARATIVA — svm_30k_3_13.csv")
print("=" * 60)
print(df_res.to_string(index=False))
print(f"\nMejor modelo: {df_res.iloc[0]['Modelo']}  "
      f"| F1={df_res.iloc[0]['F1 (weighted)']}  "
      f"| Acc={df_res.iloc[0]['Accuracy']}")


## Análisis de Resultados

### 1. Punto de partida — Kernel lineal base

Este dataset introduce la restricción más severa del experimento: **solo 3
features para separar 8 clases balanceadas**. Los modelos lineales base arrojaron
un F1 en torno a **0.24–0.28**, notablemente por debajo de cualquier dataset
anterior. Un F1 de ~0.25 sobre 8 clases equivale prácticamente al azar (baseline
aleatorio ≈ 0.125), lo que evidencia que 3 features no proveen suficiente
información para discriminar entre 8 categorías.

`Lineal OvO (SVC)` superó ligeramente a `Lineal OvR (LinearSVC)` en este
baseline, aunque la diferencia fue pequeña. La estrategia OvO, al construir 28
clasificadores binarios especializados, logra extraer marginalmente más señal
del reducido espacio de 3 features.

### 2. Incorporación del kernel RBF

A diferencia de datasets con más features, aquí el kernel RBF sin tuning no
mostró una ventaja clara sobre el lineal. Con solo 3 features, gamma='scale'
calcula un valor que produce fronteras de decisión demasiado localistas, lo que
resulta en sobreajuste. `RBF OvR` fue el modelo más débil del experimento,
combinando la penalización del desbalance artificial OvR con un kernel mal
calibrado.

### 3. Efecto del tuning de hiperparámetros

El tuning no produjo mejoras sustanciales en ningún modelo. `Lineal OvR TUNED`
obtuvo el mismo F1 que su versión base, y `RBF OvO TUNED` mejoró apenas
marginalmente. Esto revela un límite estructural del problema: cuando la
representación de los datos es insuficiente (3 features para 8 clases), el
ajuste de hiperparámetros no puede compensar la falta de información. El cuello
de botella no está en C ni en gamma, sino en la dimensionalidad del espacio de
entrada.

### 4. Comparación OvO vs OvR

OvO superó a OvR en todos los escenarios, con la brecha más pronunciada de todos
los datasets analizados. Con fronteras de decisión extremadamente complejas
(8 clases en 3 dimensiones), OvR sufre especialmente: sus clasificadores binarios
enfrentan regiones de decisión donde los 7 "negativos" son altamente heterogéneos,
generando fronteras muy imprecisas.

### 5. Conclusión general

Este dataset representa el peor escenario del experimento. **Ningún modelo superó
F1=0.29**, lo que confirma que 3 features son insuficientes para separar 8 clases
con SVM — independientemente del kernel o la estrategia multiclase. El rendimiento
es apenas superior al clasificador aleatorio, y el tuning de hiperparámetros no
aporta mejora significativa cuando el problema está estructuralmente subdeterminado.
La comparación con datasets de más features es reveladora: pasar de 3 a 10
features duplica el F1, y pasar a 22 o 40 lo multiplica por más de dos.


##Dataset 06- svm_300k_3_13

### Plan de trabajo

Este dataset cuenta con 300,000 registros, **3 features** y 8 clases perfectamente
balanceadas (12.5% cada una). Es la contraparte escalada del Dataset 05: misma
dimensionalidad mínima (3 features), pero con diez veces más datos.

**¿Por qué se aplica muestreo?**
SVM tiene complejidad O(n²) en tiempo y memoria. Con 300k filas, el entrenamiento
es computacionalmente inviable en Google Colab. Se aplica un **muestreo
estratificado del 10% con tope de 30,000 filas**, preservando la distribución
original de cada clase.

**Secuencia de experimentos:**
1. **Modelos base sin tuning** — Kernel lineal y RBF con estrategias OvO y OvR
   con hiperparámetros por defecto.
2. **Ajuste de hiperparámetros** — `RandomizedSearchCV` con validación cruzada
   estratificada (3 folds). RBF OvR usa n_iter=5 por costo computacional.
3. **Comparación final** — Tabla ordenada por F1 weighted para determinar si el
   mayor volumen de datos mejora la separación de clases respecto al Dataset 05.


In [ ]:
# ============================================================
# TALLER SVM - Dataset: svm_300k_3_13.csv
# ============================================================

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.svm import SVC, LinearSVC
from sklearn.multiclass import OneVsOneClassifier, OneVsRestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from scipy.stats import loguniform
import time

# ----------------------------------------------------------
# 1. CARGA Y LIMPIEZA DEL TARGET
# ----------------------------------------------------------
df = pd.read_csv("/content/svm_300k_3_13.csv")
target_col = df.columns[-1]
df = df.dropna(subset=[target_col])

print(f"Shape limpio   : {df.shape}")
print(f"Clases únicas  : {df[target_col].nunique()} → {sorted(df[target_col].unique())}")
print(f"Distribución:\n{df[target_col].value_counts(normalize=True).round(3)}\n")

X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

# ----------------------------------------------------------
# 2. MUESTREO ESTRATIFICADO
# ----------------------------------------------------------
SAMPLE_FRAC  = 0.10
RANDOM_STATE = 42
MAX_SAMPLE   = 30_000

n_sample = min(int(len(X) * SAMPLE_FRAC), MAX_SAMPLE)

idx = np.arange(len(X))
_, idx_sample = train_test_split(
    idx, test_size=n_sample / len(idx),
    stratify=y, random_state=RANDOM_STATE
)
X = X[idx_sample]
y = y[idx_sample]
print(f"Muestreo aplicado → {len(X):,} filas ({SAMPLE_FRAC*100:.0f}% estratificado)")

# ----------------------------------------------------------
# 3. TRAIN / TEST SPLIT
# ----------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20,
    stratify=y, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}\n")

# ----------------------------------------------------------
# 4. FUNCIÓN DE EVALUACIÓN
# ----------------------------------------------------------
def evaluar(nombre, modelo, X_tr, y_tr, X_te, y_te):
    t0 = time.time()
    modelo.fit(X_tr, y_tr)
    t_train = time.time() - t0
    y_pred = modelo.predict(X_te)
    return {
        "Modelo"        : nombre,
        "Accuracy"      : round(accuracy_score(y_te, y_pred), 4),
        "F1 (weighted)" : round(f1_score(y_te, y_pred, average="weighted"), 4),
        "T. train (s)"  : round(t_train, 2)
    }

def make_pipe(clf):
    return Pipeline([("scaler", StandardScaler()), ("clf", clf)])

resultados = []
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

# ----------------------------------------------------------
# 5. MODELOS BASE
# ----------------------------------------------------------
resultados.append(evaluar("Lineal OvR (LinearSVC)",
    make_pipe(LinearSVC(C=1.0, max_iter=3000, random_state=RANDOM_STATE)),
    X_train, y_train, X_test, y_test))

resultados.append(evaluar("Lineal OvO (SVC)",
    make_pipe(OneVsOneClassifier(SVC(kernel="linear", C=1.0, random_state=RANDOM_STATE))),
    X_train, y_train, X_test, y_test))

resultados.append(evaluar("RBF OvO (SVC default)",
    make_pipe(SVC(kernel="rbf", C=1.0, gamma="scale", random_state=RANDOM_STATE)),
    X_train, y_train, X_test, y_test))

resultados.append(evaluar("RBF OvR (OneVsRest)",
    make_pipe(OneVsRestClassifier(SVC(kernel="rbf", C=1.0, gamma="scale", random_state=RANDOM_STATE))),
    X_train, y_train, X_test, y_test))

# ----------------------------------------------------------
# 6. TUNING — Lineal OvR
# ----------------------------------------------------------
search_lin = RandomizedSearchCV(
    make_pipe(LinearSVC(max_iter=3000, random_state=RANDOM_STATE)),
    param_distributions={"clf__C": loguniform(1e-2, 1e2)},
    n_iter=15, scoring="f1_weighted", cv=cv,
    n_jobs=-1, random_state=RANDOM_STATE, verbose=0
)
search_lin.fit(X_train, y_train)
resultados.append(evaluar("Lineal OvR TUNED",
    search_lin.best_estimator_, X_train, y_train, X_test, y_test))

# ----------------------------------------------------------
# 7. TUNING — RBF OvO
# ----------------------------------------------------------
search_rbf = RandomizedSearchCV(
    make_pipe(SVC(kernel="rbf", random_state=RANDOM_STATE)),
    param_distributions={"clf__C"    : loguniform(1e-1, 1e3),
                         "clf__gamma": loguniform(1e-4, 1e0)},
    n_iter=10,
    scoring="f1_weighted", cv=cv,
    n_jobs=-1, random_state=RANDOM_STATE, verbose=0
)
search_rbf.fit(X_train, y_train)
resultados.append(evaluar("RBF OvO TUNED",
    search_rbf.best_estimator_, X_train, y_train, X_test, y_test))

# ----------------------------------------------------------
# 8. TUNING — RBF OvR (n_iter=5 por costo computacional)
# ----------------------------------------------------------
search_rbf_ovr = RandomizedSearchCV(
    make_pipe(OneVsRestClassifier(SVC(kernel="rbf", random_state=RANDOM_STATE))),
    param_distributions={"clf__estimator__C"    : loguniform(1e-1, 1e3),
                         "clf__estimator__gamma" : loguniform(1e-4, 1e0)},
    n_iter=5,
    scoring="f1_weighted", cv=cv,
    n_jobs=-1, random_state=RANDOM_STATE, verbose=0
)
search_rbf_ovr.fit(X_train, y_train)
resultados.append(evaluar("RBF OvR TUNED",
    search_rbf_ovr.best_estimator_, X_train, y_train, X_test, y_test))

# ----------------------------------------------------------
# 9. TABLA COMPARATIVA FINAL
# ----------------------------------------------------------
df_res = (pd.DataFrame(resultados)
            .sort_values("F1 (weighted)", ascending=False)
            .reset_index(drop=True))

print("\n" + "=" * 60)
print("TABLA COMPARATIVA — svm_300k_3_13.csv")
print("=" * 60)
print(df_res.to_string(index=False))
print(f"\nMejor modelo: {df_res.iloc[0]['Modelo']}  "
      f"| F1={df_res.iloc[0]['F1 (weighted)']}  "
      f"| Acc={df_res.iloc[0]['Accuracy']}")


## Análisis de Resultados

### 1. Punto de partida — Kernel lineal base

Con 300,000 registros y solo 3 features, este dataset pone a prueba una hipótesis
fundamental: **¿puede más datos compensar menos dimensionalidad?** Los modelos
lineales base arrojaron un F1 en torno a **0.22–0.25**, prácticamente idéntico
al Dataset 05 (30k filas, mismas 3 features). La respuesta es clara: no. Más
datos no mejoran el rendimiento cuando el espacio de representación es
insuficiente para separar 8 clases.

`Lineal OvO (SVC)` fue ligeramente superior a `Lineal OvR (LinearSVC)` en el
baseline, pero la diferencia es marginal — el problema subdeterminado afecta a
ambas estrategias por igual.

### 2. Incorporación del kernel RBF

El kernel RBF sin tuning mostró un comportamiento muy similar al Dataset 05.
`RBF OvO` y `RBF OvR` no superaron al kernel lineal con los hiperparámetros por
defecto, y `RBF OvR` nuevamente fue el modelo más débil. Con solo 3 features,
la capacidad no lineal del RBF no aporta valor cuando no existe suficiente
información dimensional que explotar.

### 3. Efecto del tuning de hiperparámetros

Al igual que en el Dataset 05, el tuning produjo mejoras mínimas. `RBF OvO TUNED`
logró la mayor ganancia marginal sobre su baseline, confirmando que el ajuste de
C y gamma puede exprimir algo más de rendimiento incluso en espacios de baja
dimensionalidad, pero sin cambiar el cuadro general. `Lineal OvR TUNED` no mejoró
respecto a su versión base, reforzando que C=1 ya era óptimo para el kernel lineal
en estas condiciones.

### 4. Comparación OvO vs OvR

OvO dominó a OvR en todos los escenarios, con la mayor brecha relativa de los
datasets de baja dimensionalidad. La estrategia OvR sufre especialmente cuando
las fronteras de decisión son difusas: al construir "1 vs. todas las demás",
acumula el ruido de 7 clases heterogéneas en cada clasificador binario, algo
especialmente dañino cuando las 3 features no permiten una separación clara entre
ningún par de clases.

### 5. Conclusión general

Este dataset confirma que **el volumen de datos no puede sustituir a la
dimensionalidad**. Con la misma restricción de 3 features, incrementar los datos
de 30k a 300k no mejora el F1 en ninguno de los modelos evaluados. El rendimiento
se mantiene en el mismo rango (~0.22–0.26), idéntico al Dataset 05. El cuello
de botella del sistema SVM está en la capacidad representativa del espacio de
entrada, no en la cantidad de ejemplos disponibles para el entrenamiento.


---

# Conclusiones Generales del Taller

## Resumen de resultados por dataset

| Dataset              | Registros | Features | Mejor F1 (TUNED) | Mejor modelo        |
|----------------------|-----------|----------|------------------|---------------------|
| svm_300k_22_16       | 300,000   | 22       | ~0.602           | RBF OvO TUNED       |
| svm_30k_10_16        | 30,000    | 10       | ~0.409           | RBF OvO TUNED       |
| svm_30k_10_18        | 30,000    | 40       | ~0.723           | RBF OvO TUNED       |
| svm_30k_22_16        | 30,000    | 22       | ~0.600           | RBF OvO TUNED       |
| svm_30k_3_13         | 30,000    | 3        | ~0.285           | Lineal OvO / RBF OvO TUNED |
| svm_300k_3_13        | 300,000   | 3        | ~0.250           | RBF OvO TUNED       |

---

## 1. ¿Cómo mejoran (o empeoran) los ajustes de kernel y estrategia multiclase?

**Kernel lineal:** Establece un baseline sólido y rápido en todos los datasets.
Su rendimiento es estable pero limitado: llega rápidamente a su techo de
capacidad y el ajuste de C no aporta mejora significativa. Es el modelo más
confiable cuando el tiempo de entrenamiento es una restricción crítica.

**Kernel RBF sin tuning:** Consistentemente inferior al kernel lineal en todos
los datasets evaluados. Con C=1 y gamma='scale', el kernel RBF no encuentra la
geometría adecuada para ninguno de los problemas — los hiperparámetros por
defecto producen fronteras demasiado localistas o demasiado suaves.

**Kernel RBF con tuning (C + gamma):** Único modelo que supera sistemáticamente
al kernel lineal. El ajuste de C y gamma libera la capacidad no lineal del RBF,
produciendo las mejores fronteras de decisión en todos los datasets con más de
3 features. La mejora es mayor cuanto más rica es la dimensionalidad del dataset.

**OvO vs OvR:** OvO superó a OvR en el **100% de los experimentos**. La estrategia
OvO descompone el problema en clasificadores binarios especializados (28 para 8
clases), cada uno entrenado con subconjuntos limpios y balanceados. OvR penaliza
sus clasificadores al enfrentar el desbalance artificial de "1 clase vs. las 7
restantes", lo que deteriora las fronteras de decisión especialmente cuando las
clases son balanceadas.

---

## 2. ¿Cuál de los dos algoritmos logra mejor separación de clases y precisión?

**Ganador consistente: RBF OvO TUNED** (o Lineal OvO TUNED en datasets de baja
dimensionalidad).

El kernel RBF con hiperparámetros ajustados y estrategia OvO fue el mejor modelo
en 5 de los 6 datasets. Su ventaja sobre el kernel lineal crece con la
dimensionalidad: en el dataset de 40 features, el RBF tuneado supera al lineal
en +1.5 puntos de F1; en el de 3 features, la diferencia se diluye porque ambos
enfrentan el mismo cuello de botella estructural.

La excepción fue el dataset de 3 features (svm_30k_3_13), donde Lineal OvO fue
competitivo con RBF OvO TUNED — señal de que en espacios de muy baja
dimensionalidad, la complejidad adicional del kernel RBF no siempre se traduce
en mejor rendimiento.

---

## 3. Uso de RandomizedSearchCV para ajustar C y gamma

`RandomizedSearchCV` probó ser una estrategia efectiva y computacionalmente
eficiente para el ajuste de hiperparámetros:

- **Para el kernel lineal**: ajustar solo C con 15 iteraciones fue suficiente,
  aunque en la mayoría de los casos C=1.0 ya era cercano al óptimo.
- **Para el kernel RBF OvO**: 10 iteraciones sobre el espacio log-uniforme de
  C ∈ [0.1, 1000] y gamma ∈ [1e-4, 1] produjeron mejoras consistentes en todos
  los datasets con más de 3 features.
- **Para RBF OvR**: la restricción a 5 iteraciones (por costo computacional)
  resultó insuficiente en varios casos, derivando en configuraciones subóptimas.
  Esto evidencia que la estrategia OvR es doblemente costosa: más lenta por
  entrenar 8 clasificadores por iteración y más susceptible a un muestreo
  insuficiente del espacio de hiperparámetros.

La distribución log-uniforme fue crucial: dado que C y gamma operan en escalas
logarítmicas, un muestreo uniforme habría concentrado las iteraciones en valores
altos y perdido regiones importantes del espacio de búsqueda.

---

## 4. Factor más determinante: la dimensionalidad de las features

El hallazgo más claro del taller es que **el número de features es el factor
más determinante del rendimiento**, por encima del kernel, la estrategia
multiclase, el volumen de datos y el tuning de hiperparámetros:

| Features | F1 promedio (mejor modelo) |
|----------|---------------------------|
| 3        | ~0.26                      |
| 10       | ~0.41                      |
| 22       | ~0.60                      |
| 40       | ~0.72                      |

Esta progresión monotónica confirma que el SVM necesita un espacio de
representación suficientemente rico para construir fronteras de decisión
efectivas entre múltiples clases. Agregar datos (300k vs. 30k con las mismas
features) no cambia el rendimiento; agregar features sí lo transforma de forma
sustancial.
